# PyHealth Models Tutorial — RNN Deep Dive

This notebook walks through **`pyhealth.models`** from first principles.

You will learn:
- The **`BaseModel`** contract every PyHealth model must satisfy
- How to create synthetic test data with **`create_sample_dataset()`**
- The internal architecture of **`RNNLayer`** and **`RNN`** — reading the source code line by line
- How to run forward and backward passes
- How **`MultimodalRNN`** handles mixed input types

---

In [ ]:
!pip install pyhealth

In [ ]:
import torch
import torch.nn as nn
from pyhealth.datasets import create_sample_dataset, get_dataloader
from pyhealth.models import RNN, BaseModel
from pyhealth.models.rnn import RNNLayer, MultimodalRNN

---
## Part 1: The `BaseModel` Contract

Every PyHealth model inherits from `BaseModel`, which itself inherits from both `ABC` (abstract base class) and `nn.Module` (PyTorch module).

```python
class BaseModel(ABC, nn.Module):
    def __init__(self, dataset: SampleDataset):
        ...
        self.feature_keys = list(dataset.input_schema.keys())
        self.label_keys   = list(dataset.output_schema.keys())

    def forward(self, **kwargs) -> dict[str, torch.Tensor]:
        # Subclasses implement this
        raise NotImplementedError
```

### What `BaseModel` provides

| Method | Description |
|--------|-------------|
| `device` property | Returns the device the model lives on (CPU / CUDA) |
| `get_output_size()` | Returns the FC head size (1 for binary, num_classes for multiclass) |
| `get_loss_function()` | Returns the appropriate loss: BCE for binary/multilabel, CrossEntropy for multiclass |
| `prepare_y_prob(logits)` | Applies sigmoid (binary/multilabel) or softmax (multiclass) to produce probabilities |

### The `forward()` output contract

Every `forward(**kwargs)` call returns a dict:
```python
{
    "loss":   torch.Tensor,   # scalar — backpropagatable
    "y_prob": torch.Tensor,   # predicted probabilities
    "y_true": torch.Tensor,   # ground truth labels
    "logit":  torch.Tensor,   # raw logits before activation
    "embed":  torch.Tensor,   # (optional) patient embeddings, only if embed=True in kwargs
}
```

This uniform interface means any model can plug into the `Trainer` without modification.

---
## Part 2: Creating Test Data with `create_sample_dataset()`

`create_sample_dataset()` is a convenience helper that:
1. Accepts a list of raw sample dicts
2. Fits tokenizers / processors based on the provided schema
3. Returns an `InMemorySampleDataset` (no disk I/O) ready for model instantiation

This is exactly how PyHealth's own unit tests create datasets — no MIMIC or real EHR data required.

In [ ]:
# --- Define raw samples ---
# Each sample is a dict. Keys must match the schemas below.
samples = [
    {
        "patient_id": "patient-0",
        "visit_id":   "visit-0",
        "conditions": ["E11.9",  "E11.65", "I10"],   # Type 2 DM, hypertension
        "procedures": ["99213",  "36415"],             # Office visit, blood draw
        "label": 1,
    },
    {
        "patient_id": "patient-1",
        "visit_id":   "visit-1",
        "conditions": ["E11.9"],
        "procedures": ["99213"],
        "label": 0,
    },
    {
        "patient_id": "patient-2",
        "visit_id":   "visit-2",
        "conditions": ["E11.22", "N18.3", "E11.42"],  # DM with CKD and neuropathy
        "procedures": ["99213",  "86900", "81001"],
        "label": 1,
    },
    {
        "patient_id": "patient-3",
        "visit_id":   "visit-3",
        "conditions": ["E11.65", "E11.9"],
        "procedures": ["36415"],
        "label": 0,
    },
]

# --- Define schemas ---
# Processor aliases:
#   'sequence'    → SequenceProcessor (tokenizes a list of codes to integer IDs)
#   'multi_hot'   → MultiHotProcessor (binary vector over vocabulary)
#   'timeseries'  → TimeseriesProcessor (continuous time series)
#   'tensor'      → TensorProcessor (fixed-size dense vector)
#   'binary'      → BinaryLabelProcessor (0 or 1)
input_schema  = {"conditions": "sequence", "procedures": "sequence"}
output_schema = {"label": "binary"}

# --- Create the dataset ---
dataset = create_sample_dataset(
    samples=samples,
    input_schema=input_schema,
    output_schema=output_schema,
    dataset_name="diabetes_demo",
    task_name="mortality",
)

print("Dataset type:  ", type(dataset).__name__)
print("Input schema:  ", dataset.input_schema)
print("Output schema: ", dataset.output_schema)
print("Num samples:   ", len(dataset))

In [ ]:
# Inspect the fitted processors
for key, proc in dataset.input_processors.items():
    print(f"  {key}: {type(proc).__name__}, vocab size = {proc.size()}")
print("  label:", type(dataset.output_processors['label']).__name__)

---
## Part 3: `RNNLayer` Architecture

`RNNLayer` is a low-level building block that wraps PyTorch's native RNN/LSTM/GRU with:
- **Dropout** before the recurrent computation
- **Variable-length sequence support** via pack/pad operations
- **Bidirectional support** with a down-projection to maintain hidden_size

```python
class RNNLayer(nn.Module):

    def __init__(
        self,
        input_size:   int,
        hidden_size:  int,
        rnn_type:     str   = "GRU",     # one of "RNN", "LSTM", "GRU"
        num_layers:   int   = 1,
        dropout:      float = 0.5,
        bidirectional: bool = False,
    ):
        ...
        self.dropout_layer = nn.Dropout(dropout)
        rnn_module = getattr(nn, rnn_type)       # nn.GRU, nn.LSTM, or nn.RNN
        self.rnn = rnn_module(
            input_size, hidden_size,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=bidirectional,
            batch_first=True,
        )
        if bidirectional:
            self.down_projection = nn.Linear(hidden_size * 2, hidden_size)

    def forward(
        self,
        x:    torch.Tensor,              # shape: (batch, seq_len, input_size)
        mask: Optional[torch.Tensor] = None,  # shape: (batch, seq_len), 1=valid
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        x = self.dropout_layer(x)

        # Compute actual sequence lengths from mask (or assume full sequences)
        lengths = torch.sum(mask.int(), dim=-1).cpu() if mask is not None else ...
        lengths = torch.clamp(lengths, min=1)   # avoid zero-length sequences

        # Pack → RNN → Unpack  (cuDNN optimization for variable-length batches)
        x = rnn_utils.pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)
        outputs, _ = self.rnn(x)
        outputs, _ = rnn_utils.pad_packed_sequence(outputs, batch_first=True)

        # Extract final hidden state at each sample's actual last position
        last_outputs = outputs[torch.arange(batch_size), (lengths - 1), :]

        if self.bidirectional:
            # Concatenate forward/backward final states, then project back to hidden_size
            last_outputs = self.down_projection(last_outputs)

        return outputs, last_outputs
        # outputs:      (batch, seq_len, hidden_size)  — all time steps
        # last_outputs: (batch, hidden_size)            — final hidden state
```

### Key design decisions in `RNNLayer`

1. **`pack_padded_sequence`** — tells cuDNN to skip padding positions, which is both faster and numerically correct. Without this, the RNN would process padding tokens and corrupt the final hidden state.

2. **`lengths = clamp(lengths, min=1)`** — `pack_padded_sequence` raises an error for length-0 sequences (empty visits). Clamping to 1 is a safe fallback.

3. **Bidirectional down-projection** — bidirectional outputs have `2 × hidden_size` channels; the linear layer projects back to `hidden_size` so downstream code sees a consistent dimension regardless of directionality.

In [ ]:
# --- Demonstrate RNNLayer standalone ---
batch_size  = 4
seq_len     = 10
input_size  = 32
hidden_size = 64

layer = RNNLayer(input_size=input_size, hidden_size=hidden_size, rnn_type="GRU")

x = torch.randn(batch_size, seq_len, input_size)

# Mask: batch items have different actual lengths (simulating variable-length visits)
mask = torch.zeros(batch_size, seq_len)
mask[0, :8] = 1   # 8 valid tokens
mask[1, :5] = 1   # 5 valid tokens
mask[2, :10] = 1  # all 10 valid
mask[3, :3] = 1   # 3 valid tokens
mask = mask.int()

outputs, last_outputs = layer(x, mask)

print("outputs shape (all time steps):", outputs.shape)
print("last_outputs shape (final hidden):", last_outputs.shape)

In [ ]:
# Bidirectional RNN: output shape is the same thanks to down_projection
bidi_layer = RNNLayer(input_size=input_size, hidden_size=hidden_size, bidirectional=True)
outputs_b, last_b = bidi_layer(x, mask)
print("Bidirectional outputs shape:", outputs_b.shape)   # still hidden_size after projection
print("Bidirectional last shape:   ", last_b.shape)

---
## Part 4: `RNN` Model Architecture

The `RNN` class sits one level above `RNNLayer`. It applies **separate** embedding and RNN layers for each input feature, then concatenates the final hidden states and passes them through a shared fully-connected head.

```python
class RNN(BaseModel):

    def __init__(
        self,
        dataset:       SampleDataset,
        embedding_dim: int = 128,
        hidden_dim:    int = 128,
        **kwargs              # forwarded to RNNLayer (rnn_type, num_layers, dropout, ...)
    ):
        super().__init__(dataset=dataset)

        # One embedding model shared across all features
        self.embedding_model = EmbeddingModel(dataset, embedding_dim)

        # One independent RNN layer per feature key
        self.rnn = nn.ModuleDict()
        for feature_key in self.feature_keys:
            self.rnn[feature_key] = RNNLayer(
                input_size=embedding_dim, hidden_size=hidden_dim, **kwargs
            )

        # Final FC: concatenation of all hidden states → output
        output_size = self.get_output_size()  # 1 for binary
        self.fc = nn.Linear(len(self.feature_keys) * hidden_dim, output_size)
```

### `RNN.forward()` step by step

```python
    def forward(self, **kwargs):
        patient_emb = []

        # 1. Extract value tensors and masks from each feature
        for feature_key in self.feature_keys:
            # Feature is a tuple: (value_tensor, mask_tensor, ...)
            # Schema tells us which tuple index is 'value' and which is 'mask'
            inputs[feature_key] = value
            masks[feature_key]  = mask

        # 2. Embed all features (tokenized codes → dense vectors)
        embedded = self.embedding_model(inputs, masks=masks)
        # embedded[key] shape:
        #   SequenceProcessor      → (B, seq_len, D)
        #   NestedSequenceProcessor → (B, num_visits, num_codes, D)
        #   TimeseriesProcessor    → (B, T, D)

        # 3. Handle dimensionality:
        for feature_key in self.feature_keys:
            x = embedded[feature_key]
            if x.dim() == 4:        # nested: (B, V, C, D) → sum-pool codes → (B, V, D)
                x = x.sum(dim=2)
            elif x.dim() == 2:      # static single value: (B, D) → (B, 1, D)
                x = x.unsqueeze(1)
            # Now x is always (B, T, D)

            # 4. Run per-feature RNN, take final hidden state
            _, x = self.rnn[feature_key](x, mask)
            patient_emb.append(x)  # each x: (B, hidden_dim)

        # 5. Concatenate all features' hidden states
        patient_emb = torch.cat(patient_emb, dim=1)  # (B, num_features * hidden_dim)

        # 6. Project to label space
        logits = self.fc(patient_emb)

        # 7. Compute loss, probabilities
        y_true = kwargs[self.label_key]
        loss   = self.get_loss_function()(logits, y_true)
        y_prob = self.prepare_y_prob(logits)

        return {"loss": loss, "y_prob": y_prob, "y_true": y_true, "logit": logits}
```

**Why separate RNNs per feature?** Different clinical features have different sequence semantics. Diagnosis codes have a different distributional structure than procedure codes. Separate RNNs let each feature develop its own specialized temporal representation before combining.

---
## Part 5: Instantiating and Running `RNN`

In [ ]:
# --- Instantiate the RNN model ---
model = RNN(
    dataset=dataset,
    embedding_dim=64,
    hidden_dim=64,
    # RNNLayer kwargs:
    rnn_type="GRU",       # try "LSTM" or "RNN" too
    num_layers=1,
    dropout=0.3,
    bidirectional=False,
)

print(model)
print()
print("Feature keys:",  model.feature_keys)
print("Label key:   ",  model.label_key)
print("Mode:        ",  model.mode)           # 'binary'
print("Output size: ",  model.get_output_size())  # 1
print("Device:      ",  model.device)

In [ ]:
# --- Create DataLoader and get a batch ---
train_loader = get_dataloader(dataset, batch_size=2, shuffle=False)
data_batch = next(iter(train_loader))

print("Batch keys:", list(data_batch.keys()))
for k, v in data_batch.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k}: tensor shape {v.shape}")
    elif isinstance(v, (tuple, list)):
        print(f"  {k}: tuple/list of {len(v)} items")
    else:
        print(f"  {k}: {v}")

In [ ]:
# --- Forward pass (inference mode) ---
model.eval()
with torch.no_grad():
    output = model(**data_batch)

print("Forward pass output:")
print(f"  loss:   {output['loss'].item():.4f}")
print(f"  y_prob: {output['y_prob'].squeeze().tolist()}")  # probabilities in [0, 1]
print(f"  y_true: {output['y_true'].tolist()}")
print(f"  logit:  {output['logit'].squeeze().tolist()}")

In [ ]:
# --- Requesting patient embeddings ---
# Pass embed=True to get the concatenated hidden state before the FC layer
with torch.no_grad():
    output_with_embed = model(**data_batch, embed=True)

print("With embed=True:")
print(f"  embed shape: {output_with_embed['embed'].shape}")
print(f"  Expected:    (batch_size=2, num_features={len(model.feature_keys)} × hidden_dim={model.hidden_dim} = {len(model.feature_keys) * model.hidden_dim})")
print()
print("These embeddings can be used for:")
print("  - Patient similarity search")
print("  - Visualization with UMAP / t-SNE")
print("  - Downstream tasks (transfer learning)")

In [ ]:
# --- Backward pass (training mode) ---
model.train()

# Re-fetch batch (gradients cleared)
data_batch = next(iter(train_loader))

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
optimizer.zero_grad()

output = model(**data_batch)
output["loss"].backward()   # compute gradients
optimizer.step()            # update weights

print(f"Training loss: {output['loss'].item():.4f}")
print("Backward pass completed. Weights updated.")

---
## Part 6: `MultimodalRNN` — Mixed Input Modalities

`MultimodalRNN` extends `RNN` to handle **heterogeneous inputs**. It automatically classifies each feature into:

- **Sequential** (gets its own `RNNLayer`): `SequenceProcessor`, `NestedSequenceProcessor`, `TimeseriesProcessor`, ...
- **Non-sequential** (embeddings only, no RNN): `MultiHotProcessor`, `TensorProcessor`

The architecture:
```
conditions (seq)  →  Embed  →  RNNLayer  →  hidden_cond        ┐
vitals (tensor)   →  Linear               →  embed_vitals      ├→ Concat → FC → logit
race (multi_hot)  →  Linear               →  embed_race        ┘
```

The final FC input size is `(num_sequential × hidden_dim) + (num_non_sequential × embedding_dim)`.

In [ ]:
# --- Build a multimodal dataset ---
multimodal_samples = [
    {
        "patient_id":   "p0",
        "visit_id":     "v0",
        "conditions":   ["E11.9", "I10"],         # sequential codes
        "demographics": ["female", "age_65_79"],  # multi-hot categorical
        "vitals":       [120.0, 80.0, 37.0],      # fixed-size dense vector: SBP, DBP, temp
        "label": 1,
    },
    {
        "patient_id":   "p1",
        "visit_id":     "v1",
        "conditions":   ["E11.65"],
        "demographics": ["male",   "age_50_64"],
        "vitals":       [135.0, 88.0, 36.8],
        "label": 0,
    },
    {
        "patient_id":   "p2",
        "visit_id":     "v2",
        "conditions":   ["E11.22", "N18.3"],
        "demographics": ["female", "age_80plus"],
        "vitals":       [110.0, 72.0, 37.2],
        "label": 1,
    },
    {
        "patient_id":   "p3",
        "visit_id":     "v3",
        "conditions":   ["E11.9"],
        "demographics": ["male",   "age_65_79"],
        "vitals":       [125.0, 85.0, 36.9],
        "label": 0,
    },
]

mm_dataset = create_sample_dataset(
    samples=multimodal_samples,
    input_schema={
        "conditions":   "sequence",   # will get RNNLayer
        "demographics": "multi_hot",  # embedding only
        "vitals":       "tensor",     # embedding only
    },
    output_schema={"label": "binary"},
    dataset_name="multimodal_demo",
)

print("Input processors:")
for key, proc in mm_dataset.input_processors.items():
    print(f"  {key}: {type(proc).__name__}")

In [ ]:
# --- Instantiate MultimodalRNN ---
mm_model = MultimodalRNN(
    dataset=mm_dataset,
    embedding_dim=32,
    hidden_dim=32,
    rnn_type="GRU",
)

print("Sequential features (have RNNLayer): ", mm_model.sequential_features)
print("Non-sequential features (embed only):", mm_model.non_sequential_features)
print()

# FC input dimension explained:
seq_dim    = len(mm_model.sequential_features)     * mm_model.hidden_dim
non_seq_dim = len(mm_model.non_sequential_features) * mm_model.embedding_dim
print(f"FC input: {seq_dim} (seq) + {non_seq_dim} (non-seq) = {seq_dim + non_seq_dim}")
print(f"FC out:   {mm_model.get_output_size()}  (binary)")

In [ ]:
# --- Run a forward pass ---
mm_loader = get_dataloader(mm_dataset, batch_size=4, shuffle=False)
batch = next(iter(mm_loader))

mm_model.eval()
with torch.no_grad():
    mm_out = mm_model(**batch)

print("MultimodalRNN output:")
print(f"  loss:   {mm_out['loss'].item():.4f}")
print(f"  y_prob: {mm_out['y_prob'].squeeze().tolist()}")
print(f"  y_true: {mm_out['y_true'].tolist()}")

---
## Summary

| Concept | Key API |
|---------|----------|
| Create synthetic dataset | `create_sample_dataset(samples, input_schema, output_schema)` |
| Batch iteration | `get_dataloader(dataset, batch_size=32, shuffle=True)` |
| Instantiate RNN | `RNN(dataset, embedding_dim=128, hidden_dim=128, rnn_type="GRU")` |
| Forward pass | `model(**batch)` → `{loss, y_prob, y_true, logit}` |
| Request embeddings | `model(**batch, embed=True)` → adds `embed` key |
| Backward pass | `output["loss"].backward()` |
| Mixed modalities | `MultimodalRNN(dataset, embedding_dim=128, hidden_dim=128)` |

### Choosing hyperparameters

| Hyperparameter | Guidance |
|----------------|----------|
| `rnn_type` | GRU is a good default; LSTM has more parameters but can model longer dependencies |
| `embedding_dim` | 64–256 depending on vocabulary size |
| `hidden_dim` | Usually equal to `embedding_dim`; increase for more complex patterns |
| `num_layers` | 1–2; deeper RNNs need dropout > 0 between layers |
| `dropout` | 0.3–0.5; reduces overfitting on small datasets |
| `bidirectional` | Only meaningful when the full sequence is available at inference time |